# Chapter 6 &mdash; The Product Construction: Union and Intersection

**Concept 2 of the Chapter 6 decomposition:** *The Product Construction: Union and Intersection of DFA*

Run both machines in lock-step over $Q_1\times Q_2$; union and intersection differ only in which pairs are final.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6/Concept-Product-Construction/Concept-Product-Construction.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.AnimateDFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateDFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateDFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


To combine two DFA, run them **in lock-step**. A state of the combined machine is a
**pair** $(q_1,q_2)$, and
$$\delta((q_1,q_2), a) = (\delta_1(q_1,a),\ \delta_2(q_2,a)).$$

That single construction gives **both** operations. Only the final set differs:

* **union:** $(q_1,q_2)$ final iff $q_1\in F_1$ **or** $q_2\in F_2$;
* **intersection:** final iff $q_1\in F_1$ **and** $q_2\in F_2$.

Size is at most $|Q_1|\cdot|Q_2|$, which is why pruning (Concept 3) matters.

## 2. Definitions

### Two small machines

In [ ]:
even0 = md2mc('''DFA
IF : 0 -> Od
IF : 1 -> IF
Od : 0 -> IF
Od : 1 -> Od
''')
even1 = md2mc('''DFA
IF : 1 -> Od
IF : 0 -> IF
Od : 1 -> IF
Od : 0 -> Od
''')

### The product, built by hand so the rule is visible

In [ ]:
def product_dfa(D1, D2, final):
    Q = {(a, b) for a in D1["Q"] for b in D2["Q"]}
    Dl = {((a, b), c): (step_dfa(D1, a, c), step_dfa(D2, b, c))
          for (a, b) in Q for c in D1["Sigma"]}
    F = {(a, b) for (a, b) in Q if final(a in D1["F"], b in D2["F"])}
    return mk_dfa(Q, D1["Sigma"], Dl, (D1["q0"], D2["q0"]), F)

## 3. Tests

The hand-built union agrees with `union_dfa`.

In [ ]:
U_hand = product_dfa(even0, even1, lambda x, y: x or y)
U_jove = union_dfa(even0, even1)
from itertools import product
strs = [''.join(p) for k in range(10) for p in product('01', repeat=k)]
spec = lambda s: (s.count('0') % 2 == 0) or (s.count('1') % 2 == 0)
assert all(accepts_dfa(U_hand, s) == spec(s) for s in strs)
assert all(accepts_dfa(U_jove, s) == spec(s) for s in strs)
print("union: %d states by hand, %d from union_dfa; both correct"
      % (len(U_hand["Q"]), len(U_jove["Q"])))

Intersection uses the very same transitions &mdash; only `F` changes.

In [ ]:
I_hand = product_dfa(even0, even1, lambda x, y: x and y)
I_jove = intersect_dfa(even0, even1)
spec = lambda s: (s.count('0') % 2 == 0) and (s.count('1') % 2 == 0)
assert all(accepts_dfa(I_hand, s) == spec(s) for s in strs)
assert all(accepts_dfa(I_jove, s) == spec(s) for s in strs)
print("Delta identical to the union's?", U_hand["Delta"] == I_hand["Delta"])
assert U_hand["Delta"] == I_hand["Delta"]
print("final sets differ: union %d states, intersection %d"
      % (len(U_hand["F"]), len(I_hand["F"])))

Size is the product of the sizes &mdash; before any pruning or minimizing.

In [ ]:
print("|Q1| x |Q2| = %d x %d = %d" % (len(even0["Q"]), len(even1["Q"]),
                                      len(even0["Q"]) * len(even1["Q"])))
print("product states : %d,  minimized : %d"
      % (len(I_hand["Q"]), len(min_dfa(I_hand)["Q"])))

## 4. Animation

Four pair-states, and every edge moves both coordinates at once.

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(min_dfa(intersect_dfa(even0, even1)), FuseEdges=True)

## 5. Exercises


1. Build "symmetric difference" with the same construction. Which `final` do you pass?
2. Why is the product construction impossible for PDA intersection? (Chapter 12.)
3. Combine a 3-state and a 5-state DFA. How large is the product, before minimizing?

In [ ]:
# Your work for the exercises above.